## Day 1: Classifying Book Cover Photos With a Pretrained Multimodal Model

A used bookstore photographs every incoming book. Instead of training a CNN
to recognize genre/condition from scratch, we call a pretrained
multimodal (vision-language) model and ask it directly. Below is a
**mock-first** wrapper: it works with no API key at all (returns a canned
response) and would call a real vision API once a key is provided.


In [ ]:
import os

def classify_book_photo(image_path: str, api_key: str | None = None) -> dict:
    """Classify a photo of a book (genre guess + condition guess).

    Falls back to a deterministic mock response when no API key is set,
    so this function is safe to call in a notebook with no network access.
    """
    api_key = api_key or os.getenv("VISION_API_KEY")
    if not api_key:
        return {
            "genre_guess": "science fiction",
            "condition_guess": "good (minor shelf wear)",
            "source": "mock",
        }
    # A real implementation would send image bytes/URL + a prompt to a
    # vision-capable API and parse its response here.
    return call_multimodal_api(image_path, api_key=api_key)

result = classify_book_photo("cover_photos/dune_1965_paperback.jpg")
result


**Cost/time contrast:** training a genre-and-condition classifier from
scratch would need thousands of labeled cover photos and real training
time. The mock-first call above costs nothing to develop against, and a
real deployment costs one inference call per photo.


## Day 2: Extracting Structured JSON From a Packing Slip Photo

Incoming book shipments arrive with a paper packing slip. We prompt a
multimodal model for **structured JSON** (not a free-text description),
then defensively parse whatever text comes back with a `safe_json()`
helper, since LLM output is not guaranteed to be valid JSON.


In [ ]:
import json

def safe_json(raw_text: str, default=None):
    """Parse model output as JSON, tolerating markdown code fences.

    Returns `default` instead of raising if parsing fails.
    """
    cleaned = raw_text.strip().removeprefix("```json").removesuffix("```").strip()
    try:
        return json.loads(cleaned)
    except (json.JSONDecodeError, TypeError):
        return default

# Simulated raw model output for a packing slip photo (note the code fence
# a real model sometimes adds around JSON):
raw_model_output = '''```json
{
  "supplier": "Riverside Book Distributors",
  "shipment_date": "2026-08-14",
  "titles": ["Dune", "Foundation", "The Left Hand of Darkness"],
  "box_count": 3
}
```'''

parsed = safe_json(raw_model_output, default={})
parsed


**Object detection contrast:** a YOLO-style detector could count how many
boxes sit on a loading-dock photo (bounding boxes + labels), but it has no
concept of "supplier name" or "shipment date" — that requires prompted,
semantic extraction like `safe_json` above, not object detection.

**PII note:** packing slips often include a customer shipping label
(name, home address). Mask or drop those fields before persisting the
parsed JSON to a shared database.


## Day 3: Summarizing a Multi-Page Book Condition Report

An appraiser's PDF condition report can run 10+ pages. We extract text
with `pypdf`, fall back to a multimodal "read this page" call for any
page that's a scanned image with no text layer, summarize long reports in
map-reduce chunks, and finally check that any dollar figure in the
summary is actually grounded in the source text.


In [ ]:
# from pypdf import PdfReader
#
# reader = PdfReader('dune_first_edition_appraisal.pdf')
# pages_text = [page.extract_text() or '' for page in reader.pages]

# Simulated extracted page text (stands in for pypdf output above):
pages_text = [
    "Item: Dune, first edition, 1965 Chilton Books hardcover.",
    "Condition: spine creased, dust jacket present with light foxing.",
    "Appraised value: $1,850 based on comparable recent sales.",
]

def summarize_chunk(text: str) -> str:
    # Placeholder for a real LLM summarization call.
    return text[:60] + ("..." if len(text) > 60 else "")

def summarize_long_document(pages: list[str]) -> str:
    chunk_summaries = [summarize_chunk(p) for p in pages]     # map
    return " ".join(chunk_summaries)                          # reduce (simplified)

final_summary = summarize_long_document(pages_text)
final_summary


In [ ]:
import re

def numbers_in_summary_are_grounded(summary: str, source_text: str) -> bool:
    """Anti-hallucination check: every number cited in the summary
    must appear somewhere in the original source text.
    """
    cited_numbers = re.findall(r"\d+(?:\.\d+)?", summary)
    return all(num in source_text for num in cited_numbers)

source_text = " ".join(pages_text)
grounded = numbers_in_summary_are_grounded("Appraised value: $1,850", source_text)
grounded


## Day 4: Scraping an Auction Listings Table Into a CSV Report

A rare-book auction site publishes current listings as an HTML table.
We parse it with BeautifulSoup, load the rows into a dataframe, sort by
price, and export a CSV — no AI model needed for this step at all.


In [ ]:
from bs4 import BeautifulSoup
import pandas as pd

# Simulated page HTML (stands in for a real requests.get(...).text):
sample_html = '''
<table class="listings">
  <tr><th>lot</th><th>title</th><th>current_bid</th></tr>
  <tr><td>101</td><td>Dune, 1965 1st ed.</td><td>1200</td></tr>
  <tr><td>102</td><td>Foundation, 1951 1st ed.</td><td>950</td></tr>
  <tr><td>103</td><td>Neuromancer, signed</td><td>640</td></tr>
</table>
'''

soup = BeautifulSoup(sample_html, "html.parser")
table = soup.find("table", class_="listings")
rows = table.find_all("tr")[1:]  # skip header row

records = []
for row in rows:
    lot, title, bid = [td.get_text(strip=True) for td in row.find_all("td")]
    records.append({"lot": int(lot), "title": title, "current_bid": float(bid)})

df = pd.DataFrame(records)
report = df.sort_values("current_bid", ascending=False)
report


In [ ]:
# report.to_csv('auction_listings_report.csv', index=False)

# Scraping etiquette (not executed here, illustrative):
# - check https://<site>/robots.txt before scraping any path
# - pace requests, e.g. time.sleep(1) between page fetches
# - wrap each page fetch in try/except and collect failures
#   instead of aborting the whole batch on one bad page
print('report ready:', len(report), 'rows')
